# ISIC 2024 - 多模態網路（圖片 CNN ＋ 表格 MLP，端到端訓練）

一個神經網路同時看**圖片**與**metadata**：

```
圖片 ──► CNN 骨幹（EfficientNet-B0 / ResNet-18）──► 影像特徵 ─┐
                                                               ├─► concat ─► FC ─► 惡性機率
metadata ──► MLP（Linear-BN-ReLU-Dropout ×2）────► 表格特徵 ───┘
```

跟「LightGBM ＋ 圖片預測當特徵」（兩階段 stacking）相比，這是**純深度學習**的融合：兩個分支一起反向傳播。

這是**組員 E（輸入資料與融合）**的實驗 `E4`，其他設定與共同基準 `BASE` 相同，可直接和 `BASE`（只用圖片）比較。

**Kaggle 設定**：GPU、Internet 打開。不用提交，輸出 `exp_E4.json`、`oof_E4.csv`。

In [ ]:
import glob, io, json, os, time
import numpy as np
import pandas as pd
import h5py
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

KAGGLE_INPUT = '/kaggle/input'

def find_dir(fname):
    # Kaggle 的掛載路徑會變（例如多一層 competitions/），所以用搜尋的找檔案所在資料夾
    for depth in range(1, 5):
        hits = sorted(glob.glob(os.path.join(KAGGLE_INPUT, *['*'] * depth, fname)),
                      key=lambda p: ('isic-2024-challenge' not in p, p))  # 優先選比賽本身的資料
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'在 {KAGGLE_INPUT} 底下找不到 {fname}，請確認右側 Input 有加入')

DATA = find_dir('train-metadata.csv')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
os.makedirs('figures', exist_ok=True)
print('資料路徑:', DATA, '| device:', DEVICE)

In [ ]:
EXP, member, variable = 'E4', 'E', '多模態融合'
# 與共同基準 BASE 相同，只是多了 metadata 分支
CFG = dict(model='efficientnet_b0', img=128, optimizer='adamw', lr=3e-4, aug='basic', dropout=0.3,
           neg_ratio=20, loss='bce', pos_weight=1.0, epochs=10, patience=3, bs=128,
           eval_neg=20000, workers=4, pretrained=True)

In [ ]:
def pauc_above_tpr(y_true, y_pred, min_tpr=0.80):
    # 官方評分：ROC 曲線在 TPR >= 0.8 的面積，範圍 0 ~ 0.2
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = 1.0 - np.asarray(y_pred)
    max_fpr = 1 - min_tpr
    scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    return 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (scaled - 0.5)

def make_folds(train):
    # 跟 LightGBM、MLP 完全相同的切法，融合時才不會洩漏
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    return list(sgkf.split(train, train['target'], train['patient_id']))

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def make_tfms(img, aug='basic'):
    """aug: none（不增強）/ basic（翻轉、旋轉、色彩）/ strong（再加隨機裁切縮放、仿射、模糊）"""
    norm = [T.ToTensor(), T.Normalize(MEAN, STD)]
    if aug == 'none':
        train_ops = [T.Resize((img, img))]
    elif aug == 'basic':
        train_ops = [T.Resize((img, img)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                     T.RandomRotation(90), T.ColorJitter(0.2, 0.2, 0.1, 0.02)]
    elif aug == 'strong':
        train_ops = [T.RandomResizedCrop(img, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                     T.RandomRotation(180), T.RandomAffine(0, translate=(0.1, 0.1), shear=10),
                     T.ColorJitter(0.3, 0.3, 0.2, 0.05), T.RandomApply([T.GaussianBlur(3)], p=0.3)]
    else:
        raise ValueError(aug)
    eval_tfm = T.Compose([T.Resize((img, img))] + norm)
    return T.Compose(train_ops + norm), eval_tfm


class LesionDataset(Dataset):
    """從 hdf5 讀 JPEG bytes。每個 worker 各自開檔，避免 h5py 多程序問題。meta 是選用的表格特徵。"""
    def __init__(self, h5_path, ids, targets=None, tfm=None, meta=None):
        self.h5_path, self.ids, self.targets, self.tfm, self.meta = h5_path, list(ids), targets, tfm, meta
        self.h5 = None
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, 'r')
        img = Image.open(io.BytesIO(bytes(self.h5[self.ids[i]][()]))).convert('RGB')
        x = self.tfm(img)
        y = torch.tensor(-1.0 if self.targets is None else float(self.targets[i]))
        m = torch.zeros(0) if self.meta is None else torch.from_numpy(self.meta[i])
        return x, m, y


@torch.no_grad()
def predict(model, h5_path, ids, tfm, meta=None):
    dl = DataLoader(LesionDataset(h5_path, ids, tfm=tfm, meta=meta), batch_size=CFG['bs'] * 2,
                    shuffle=False, num_workers=CFG['workers'])
    model.eval()  # BatchNorm 用累積的統計量、Dropout 關閉
    out = []
    for x, m, _ in dl:
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            logit = model(x.to(DEVICE), m.to(DEVICE))
        out.append(torch.sigmoid(logit.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

In [ ]:
class SimpleCNN(nn.Module):
    """自己搭的 CNN（對應上課 W5）：[Conv3x3 → BN → ReLU] ×2 → MaxPool，重複 4 次 → GAP → Dropout → FC"""
    def __init__(self, n_out=1):
        super().__init__()
        layers, c_in = [], 3
        for c in [32, 64, 128, 256]:
            layers += [nn.Conv2d(c_in, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(c_in, n_out))
        self.out_dim = c_in
    def forward_features(self, x):
        return self.pool(self.features(x)).flatten(1)
    def forward(self, x):
        return self.head(self.pool(self.features(x)))


def build_backbone(name, pretrained):
    """回傳 (去掉分類頭的骨幹, 特徵維度)。圖片分類模型和多模態模型共用。"""
    if name == 'simple_cnn':
        m = SimpleCNN()
        m.forward = m.forward_features
        return m, m.out_dim
    if name == 'resnet18':
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        d = m.fc.in_features; m.fc = nn.Identity()
        return m, d
    if name == 'efficientnet_b0':
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        d = m.classifier[1].in_features; m.classifier = nn.Identity()
        return m, d
    if name == 'vit_tiny':
        import timm  # Kaggle 內建；Vision Transformer（對應上課 W8）
        m = timm.create_model('vit_tiny_patch16_224', pretrained=pretrained, num_classes=0, img_size=CFG['img'])
        return m, m.num_features
    raise ValueError(name)


class ImageClassifier(nn.Module):
    def __init__(self, name, pretrained, dropout=0.3):
        super().__init__()
        self.backbone, d = build_backbone(name, pretrained)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, 1))
    def forward(self, x, meta=None):
        return self.head(self.backbone(x)).squeeze(1)

In [ ]:
class MultiModalNet(nn.Module):
    def __init__(self, backbone, n_meta, pretrained):
        super().__init__()
        self.cnn, d_img = build_backbone(backbone, pretrained)
        self.mlp = nn.Sequential(nn.Linear(n_meta, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3))
        self.head = nn.Sequential(nn.Linear(d_img + 64, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x, meta):
        return self.head(torch.cat([self.cnn(x), self.mlp(meta)], dim=1)).squeeze(1)

In [ ]:
class FocalLoss(nn.Module):
    """Focal loss：降低「已經很容易判斷」樣本的權重，讓模型專注在難的樣本（處理不平衡）"""
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha
    def forward(self, logits, y):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, y, reduction='none')
        p_t = torch.exp(-bce)
        a_t = self.alpha * y + (1 - self.alpha) * (1 - y)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


def make_loss():
    if CFG['loss'] == 'bce':
        return nn.BCEWithLogitsLoss()
    if CFG['loss'] == 'pos_weight':  # 惡性樣本的 loss 乘上權重
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(CFG['pos_weight']), device=DEVICE))
    if CFG['loss'] == 'focal':
        return FocalLoss()
    raise ValueError(CFG['loss'])


def make_optimizer(params):
    if CFG['optimizer'] == 'sgd':
        return torch.optim.SGD(params, lr=CFG['lr'], momentum=0.9, weight_decay=1e-4)
    if CFG['optimizer'] == 'adam':
        return torch.optim.Adam(params, lr=CFG['lr'])
    return torch.optim.AdamW(params, lr=CFG['lr'], weight_decay=1e-2)


def train_one_fold(fold, tr, va, build_fn, meta=None):
    """每個 epoch：全部惡性 + 重新抽樣的良性；在驗證子集上算 pAUC 做 early stopping。"""
    rng = np.random.default_rng(SEED + fold)
    torch.manual_seed(SEED + fold)
    train_tfm, eval_tfm = make_tfms(CFG['img'], CFG['aug'])
    y_all = train.target.values
    pos_idx, neg_idx = tr[y_all[tr] == 1], tr[y_all[tr] == 0]
    # 驗證子集：驗證 fold 的全部惡性 + 隨機良性（加快每個 epoch 的評估）
    va_pos, va_neg = va[y_all[va] == 1], va[y_all[va] == 0]
    va_sub = np.concatenate([va_pos, rng.choice(va_neg, min(len(va_neg), CFG['eval_neg']), replace=False)])

    model = build_fn().to(DEVICE)
    opt = make_optimizer(model.parameters())
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'])
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    loss_fn = make_loss()
    best, best_state, wait, hist = -1, None, 0, []
    for ep in range(CFG['epochs']):
        n_neg = min(len(neg_idx), len(pos_idx) * CFG['neg_ratio'])
        idx = np.concatenate([pos_idx, rng.choice(neg_idx, n_neg, replace=False)])
        ds = LesionDataset(H5_TRAIN, train.isic_id.values[idx], y_all[idx], train_tfm,
                           None if meta is None else meta[idx])
        dl = DataLoader(ds, batch_size=CFG['bs'], shuffle=True, num_workers=CFG['workers'], drop_last=len(idx) > CFG['bs'])
        model.train()
        tot, n = 0.0, 0
        for x, m, y in dl:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
                loss = loss_fn(model(x, m), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); n += len(y)
        sched.step()
        p = predict(model, H5_TRAIN, train.isic_id.values[va_sub], eval_tfm, None if meta is None else meta[va_sub])
        y_sub = y_all[va_sub]
        score = pauc_above_tpr(y_sub, p)
        val_loss = float(nn.functional.binary_cross_entropy(torch.tensor(p, dtype=torch.float32).clamp(1e-7, 1 - 1e-7),
                                                            torch.tensor(y_sub, dtype=torch.float32)))
        hist.append(dict(fold=fold, epoch=ep, train_loss=tot / max(n, 1), val_loss=val_loss, val_pauc=score))
        print(f'  fold {fold} epoch {ep}: train_loss {tot / max(n, 1):.4f}  val_loss {val_loss:.4f}  val_pAUC(subset) {score:.4f}')
        if score > best:
            best, wait = score, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= CFG['patience']:
                print(f'  early stop at epoch {ep}')
                break
    model.load_state_dict(best_state)
    oof_va = predict(model, H5_TRAIN, train.isic_id.values[va], eval_tfm, None if meta is None else meta[va])
    return model, oof_va, hist


def plot_history(hist, name):
    h = pd.DataFrame(hist)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for f, d in h.groupby('fold'):
        axes[0].plot(d.epoch, d.train_loss, label=f'fold {f}')
        axes[1].plot(d.epoch, d.val_pauc, label=f'fold {f}')
    axes[0].set_title(f'{name}: train loss'); axes[1].set_title(f'{name}: validation pAUC (subset)')
    for ax in axes:
        ax.set_xlabel('epoch'); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.savefig(f'figures/curves_{name}.png', dpi=150, bbox_inches='tight'); plt.show()

## metadata 特徵（與 LightGBM 相同）＋ 每折標準化

In [ ]:
train = pd.read_csv(f'{DATA}/train-metadata.csv', low_memory=False)
test_cols = pd.read_csv(f'{DATA}/test-metadata.csv', nrows=1).columns
H5_TRAIN = f'{DATA}/train-image.hdf5'
drop = {'isic_id', 'patient_id', 'target', 'image_type', 'attribution', 'copyright_license'}
common = [c for c in test_cols if c in train.columns and c not in drop]
cat_cols = [c for c in common if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in common if c not in cat_cols]
pn_cols = [c for c in num_cols if c not in ('age_approx', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z')]
g = train.groupby('patient_id')[pn_cols]
z = (train[pn_cols] - g.transform('mean')) / (g.transform('std') + 1e-6)
z.columns = [c + '_pnorm' for c in pn_cols]
train = pd.concat([train, z], axis=1)
train['patient_n_lesions'] = train.groupby('patient_id')['isic_id'].transform('count')
num_all = num_cols + list(z.columns) + ['patient_n_lesions']
X_num = train[num_all].astype('float32').to_numpy()
X_num[~np.isfinite(X_num)] = np.nan
X_cat = pd.get_dummies(train[cat_cols].astype('category'), dummy_na=True).astype('float32').to_numpy()
folds = make_folds(train)
print('metadata 維度：', X_num.shape[1] + X_cat.shape[1])

def meta_for_fold(tr):
    # 只用訓練 fold 的統計量：補中位數、標準化、截斷
    med = np.nanmedian(X_num[tr], axis=0); med = np.where(np.isfinite(med), med, 0)
    X = np.where(np.isnan(X_num), med, X_num)
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
    return np.hstack([np.clip((X - mu) / sd, -5, 5), X_cat]).astype('float32')

## 5 折訓練

In [ ]:
oof = np.zeros(len(train))
all_hist, fold_scores = [], []
t_all = time.time()
for fold, (tr, va) in enumerate(folds):
    t0 = time.time()
    meta = meta_for_fold(tr)
    model, oof[va], hist = train_one_fold(fold, tr, va, lambda: MultiModalNet(CFG['model'], meta.shape[1], CFG['pretrained']), meta)
    torch.save(model.state_dict(), f'{EXP}_fold{fold}.pt')
    all_hist += hist
    fold_scores.append(pauc_above_tpr(train.target.values[va], oof[va]))
    print(f'fold {fold}: pAUC = {fold_scores[-1]:.4f}  ({time.time() - t0:.0f}s)')

total = pauc_above_tpr(train.target, oof)
minutes = (time.time() - t_all) / 60
print(f'\n實驗 {EXP}（多模態：{CFG["model"]} + MLP）OOF pAUC = {total:.4f}')
pd.DataFrame({'isic_id': train.isic_id, 'pred': oof}).to_csv(f'oof_{EXP}.csv', index=False)
pd.DataFrame(all_hist).to_csv(f'history_{EXP}.csv', index=False)
result = dict(exp=EXP, member=member, variable=variable, change={'input': 'image + metadata'}, kind='multimodal',
              cfg=CFG, oof_pauc=total, fold_pauc=fold_scores, minutes=minutes)
json.dump(result, open(f'exp_{EXP}.json', 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
plot_history(all_hist, EXP)